In [ ]:
from multiprocessing import freeze_support
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import DenseNet121_Weights, EfficientNet_B0_Weights
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int):
    worker_seed = 42 + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# -----------------------------
# Model Components
# -----------------------------
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.mean(dim=(2, 3))
        y = self.fc1(y)
        y = self.relu(y)
        y = self.fc2(y)
        y = self.sigmoid(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels)
        )
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_pool = x.mean(dim=(2, 3))
        max_pool = torch.amax(x, dim=(2, 3))
        ca = self.mlp(avg_pool) + self.mlp(max_pool)
        ca = torch.sigmoid(ca).view(x.size(0), x.size(1), 1, 1)
        x = x * ca

        avg_out = x.mean(dim=1, keepdim=True)
        max_out = torch.amax(x, dim=1, keepdim=True)
        sa = torch.cat([avg_out, max_out], dim=1)
        sa = self.sigmoid(self.conv(sa))
        x = x * sa
        return x


def unfreeze_last_n_params(model: nn.Module, n: int):
    for p in model.parameters():
        p.requires_grad = False
    params = list(model.parameters())
    for p in params[-n:]:
        p.requires_grad = True


class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        # ✅ Updated with new weights parameter (no more warnings)
        self.densenet = models.densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        self.densenet.classifier = nn.Identity()
        unfreeze_last_n_params(self.densenet, n=50)

        self.se_d = SEBlock(1024)
        self.head_d = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512),
            nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(512, 256), nn.BatchNorm1d(256),
            nn.Dropout(0.3), nn.ReLU()
        )

        # ✅ Updated EfficientNet
        self.efficientnet = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.efficientnet.classifier = nn.Identity()
        for p in self.efficientnet.parameters():
            p.requires_grad = False

        self.se_e = SEBlock(1280)
        self.head_e = nn.Sequential(
            nn.Linear(1280, 512), nn.BatchNorm1d(512),
            nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(512, 256), nn.BatchNorm1d(256),
            nn.Dropout(0.3), nn.ReLU()
        )

        fusion_dim = 256 + 256
        self.cbam = CBAM(fusion_dim)
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        f_d = self.densenet(x)
        f_d = f_d.unsqueeze(-1).unsqueeze(-1)
        f_d = self.se_d(f_d).squeeze(-1).squeeze(-1)
        f_d = self.head_d(f_d)

        f_e = self.efficientnet(x)
        f_e = f_e.unsqueeze(-1).unsqueeze(-1)
        f_e = self.se_e(f_e).squeeze(-1).squeeze(-1)
        f_e = self.head_e(f_e)

        f = torch.cat([f_d, f_e], dim=1)
        f = f.unsqueeze(-1).unsqueeze(-1)
        f = self.cbam(f).squeeze(-1).squeeze(-1)
        out = self.classifier(f)
        return out


# -----------------------------
# MAIN FUNCTION
# -----------------------------
def main():
    set_seed(42)

    input_path = "/kaggle/input/datasets/sabbir4724/sabbirs-dataset/BDCXR"
    output_dir = "/kaggle/working/"
    os.makedirs(output_dir, exist_ok=True)

    EPOCHS = 30
    BATCH_SIZE = 64
    LR = 1e-4
    IMG_SIZE = (224, 224)
    K_FOLDS = 3

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_gpus = torch.cuda.device_count()
    print(f"✅ Available GPUs: {num_gpus} | Device: {device}")

    transform = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_dataset = datasets.ImageFolder(input_path, transform=transform)
    num_classes = len(full_dataset.classes)
    class_names = full_dataset.classes

    print(f"Classes: {class_names} | Total images: {len(full_dataset)}")

    labels = [sample[1] for sample in full_dataset.samples]
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        print(f"\n{'='*40} FOLD {fold + 1}/{K_FOLDS} {'='*40}")

        fold_output_dir = os.path.join(output_dir, f"fold_{fold+1}")
        os.makedirs(fold_output_dir, exist_ok=True)
        best_model_file = os.path.join(fold_output_dir, "best_model.pth")

        np.random.seed(42)
        np.random.shuffle(train_idx)
        n_train = int(0.7 * len(train_idx))
        train_indices = train_idx[:n_train]
        val_indices = train_idx[n_train:]
        test_indices = test_idx

        train_set = Subset(full_dataset, train_indices)
        val_set = Subset(full_dataset, val_indices)
        test_set = Subset(full_dataset, test_indices)

        g = torch.Generator()
        g.manual_seed(42)

        train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, 
                                  pin_memory=True, worker_init_fn=seed_worker, generator=g)
        val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, 
                                pin_memory=True, worker_init_fn=seed_worker, generator=g)
        test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, 
                                 pin_memory=True, worker_init_fn=seed_worker, generator=g)

        # Model
        model = FusionModel(num_classes)
        if num_gpus > 1:
            model = nn.DataParallel(model)
        model = model.to(device)

        # Class weights
        train_labels_fold = [full_dataset.samples[i][1] for i in train_indices]
        class_weights = compute_class_weight("balanced", classes=np.unique(train_labels_fold), y=train_labels_fold)
        class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
        optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

        best_acc = 0.0
        best_epoch = -1
        history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

        for epoch in range(1, EPOCHS + 1):
            model.train()
            running_loss = running_corrects = 0.0
            for X, y in train_loader:
                X, y = X.to(device), y.to(device)
                optimizer.zero_grad()
                outputs = model(X)
                loss = criterion(outputs, y)
                loss.backward()
                optimizer.step()

                preds = outputs.argmax(dim=1)
                running_loss += loss.item() * X.size(0)
                running_corrects += (preds == y).sum().item()

            train_loss = running_loss / len(train_set)
            train_acc = running_corrects / len(train_set)

            model.eval()
            val_loss = val_corrects = 0.0
            with torch.no_grad():
                for X, y in val_loader:
                    X, y = X.to(device), y.to(device)
                    outputs = model(X)
                    loss = criterion(outputs, y)
                    preds = outputs.argmax(dim=1)
                    val_loss += loss.item() * X.size(0)
                    val_corrects += (preds == y).sum().item()

            val_loss /= len(val_set)
            val_acc = val_corrects / len(val_set)

            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)

            print(f"Epoch {epoch:2d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

            if val_acc > best_acc:
                best_acc = val_acc
                best_epoch = epoch
                state_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
                torch.save(state_dict, best_model_file)

        # ====================== EVALUATION ======================
        model = FusionModel(num_classes)  # Fresh model
        state_dict = torch.load(best_model_file, map_location=device)
        model.load_state_dict(state_dict)
        
        if num_gpus > 1:
            model = nn.DataParallel(model)
        model = model.to(device)
        model.eval()

        y_true, y_pred = [], []
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                outputs = model(X)
                preds = outputs.argmax(dim=1)
                y_true.extend(y.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        accuracy = np.mean(np.array(y_true) == np.array(y_pred))
        report = classification_report(y_true, y_pred, target_names=class_names, digits=4)

        print(f"\nFold {fold+1} Test Accuracy: {accuracy:.4f}")
        print(report)

        fold_results.append(accuracy)

        # Save results
        pd.DataFrame(history).to_csv(os.path.join(fold_output_dir, "training_history.csv"), index=False)
        
        with open(os.path.join(fold_output_dir, "performance_metrics.txt"), "w") as f:
            f.write(f"FOLD {fold+1} RESULTS\n")
            f.write(f"Best Epoch: {best_epoch}\n")
            f.write(f"Best Val Acc: {best_acc:.6f}\n")
            f.write(f"Test Accuracy: {accuracy:.6f}\n\n")
            f.write(report)

        # High-Resolution Plots
        plt.figure(figsize=(10, 6), dpi=300)
        plt.plot(history["train_acc"], label="Train Accuracy")
        plt.plot(history["val_acc"], label="Validation Accuracy")
        plt.scatter([best_epoch-1], [best_acc], label="Best", marker="o", s=120, color='red')
        plt.title(f"Accuracy Curve - Fold {fold+1}", fontsize=16)
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(fold_output_dir, "accuracy_curve.png"), dpi=400, bbox_inches='tight')
        plt.close()

        plt.figure(figsize=(10, 6), dpi=300)
        plt.plot(history["train_loss"], label="Train Loss")
        plt.plot(history["val_loss"], label="Validation Loss")
        plt.title(f"Loss Curve - Fold {fold+1}", fontsize=16)
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(fold_output_dir, "loss_curve.png"), dpi=400, bbox_inches='tight')
        plt.close()

        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(10, 8), dpi=300)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Confusion Matrix - Fold {fold+1}', fontsize=16)
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.savefig(os.path.join(fold_output_dir, "confusion_matrix.png"), dpi=400, bbox_inches='tight')
        plt.close()

        print(f"✅ Fold {fold+1} completed successfully.\n")

    # Final Summary
    mean_acc = np.mean(fold_results)
    std_acc = np.std(fold_results)
    print(f"\n{'='*60}")
    print(f"FINAL 3-FOLD CROSS VALIDATION SUMMARY")
    print(f"Mean Test Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
    print(f"{'='*60}")

    with open(os.path.join(output_dir, "kfold_summary.txt"), "w") as f:
        f.write(f"3-Fold CV Results (Seed=42 | 2x T4 GPU)\n")
        f.write(f"Mean Accuracy: {mean_acc:.6f} ± {std_acc:.6f}\n")
        for i, acc in enumerate(fold_results):
            f.write(f"Fold {i+1}: {acc:.6f}\n")

    print(f"\n🎉 All results saved in /kaggle/working/")


if __name__ == "__main__":
    freeze_support()
    main()
